# Converting schema to clean human-readable version

## Read the schema (of unmapped entity)

In [33]:
import yaml
from typing import Any, Dict

In [34]:
## load the schema from the YAML file

path_schema = r'schema_unmapped.yaml'  
with open(path_schema, 'r') as f:
    schema = yaml.safe_load(f)
    
schema  # Now a Python dict

{'$schema': 'http://json-schema.org/draft-07/schema#',
 'title': 'UnmappedEntity',
 'type': 'object',
 'description': 'Staging schema for raw, unstructured data ingestion. All fields capture free-text inputs before data normalization.',
 'required': ['technology_name'],
 'properties': {'technology_name': {'type': 'string',
   'description': 'The literal unparsed common name or label of the technology as specified in the raw source material.'},
  'technology': {'type': 'object',
   'description': 'Contains raw descriptive properties detailing the engineering, structural, and hardware context of the unmapped entity.',
   'properties': {'technology_description': {'type': 'string',
     'description': 'Free-text engineering parameters describing the technical baseline of the asset.'},
    'technology_type': {'type': 'string',
     'description': 'The structural archetype classifying how the physical asset interacts with energy streams.',
     'examples': ['conversion', 'storage', 'distribu

In [35]:
def determine_type_prefix(field_info):
    """Returns the bracketed type prefix string using Pythonic naming conventions."""
    field_type = field_info.get("type", "any")
    if field_type == "object":
        return "Dictionary"
    elif field_type == "array":
        item_type = field_info.get("items", {}).get("type", "any")
        if item_type == "object":
            return "List of Dictionaries"
        return f"List of {item_type.capitalize()}s]"
    return f"{field_type.capitalize()}"


def parse_properties(schema, required_fields, indent_level=4):
    """Recursively processes properties into a flat blueprint, inserting requirements into the description text."""
    lines = []
    space = " " * indent_level
    properties = schema.get("properties", {})

    for field_name, field_info in properties.items():
        field_type = field_info.get("type", "any")
        description = field_info.get("description", "No description provided.")
        
        # 1. Determine requirement status prefix
        req_prefix = "Required; " if field_name in required_fields else ""
        
        # 2. Determine pythonic data type prefix
        type_prefix = determine_type_prefix(field_info)

        # 3. Assemble the full text string
        desc_text = f"[{req_prefix}{type_prefix}] {description}"
        if "examples" in field_info:
            desc_text += f" Suggested: {field_info['examples']}"

        # Handle Nested Dictionaries
        if field_type == "object" and "properties" in field_info:
            inner_required = field_info.get("required", [])
            lines.append(f"{space}{field_name}: # {desc_text}")
            lines.extend(
                parse_properties(field_info, inner_required, indent_level + 4)
            )

        # Handle Lists of Dictionaries
        elif field_type == "array" and "properties" in field_info.get("items", {}):
            inner_required = field_info["items"].get("required", [])
            lines.append(f"{space}{field_name}: # {desc_text}")
            lines.extend(
                parse_properties(field_info["items"], inner_required, indent_level + 4)
            )

        # Handle primitive data types (Strings, Numbers, etc.)
        else:
            lines.append(f'{space}{field_name}: "{desc_text}"')

    return lines


def convert_schema_to_clean_blueprint(input_yaml_path, output_yaml_path):
    """Reads a JSON Schema and outputs a clean, pythonic comment-style documentation blueprint."""
    with open(input_yaml_path, "r", encoding="utf-8") as f:
        schema_data = yaml.safe_load(f)

    title = schema_data.get("title", "SchemaDocumentation")
    global_desc = schema_data.get("description", "No global description provided.")
    top_required = schema_data.get("required", [])

    output_lines = [
        f"{title}: # {global_desc}",
    ]

    body_lines = parse_properties(schema_data, top_required, indent_level=4)
    output_lines.extend(body_lines)

    with open(output_yaml_path, "w", encoding="utf-8") as f:
        f.write("\n".join(output_lines) + "\n")

    print(f"Successfully generated clean human blueprint at: {output_yaml_path}")

In [36]:
input_file = "schema_unmapped.yaml"
output_file = "human_documentation.yaml"
print(
    f"Point the script to your '{input_file}' to generate '{output_file}'."
)
convert_schema_to_clean_blueprint(input_file, output_file)

Point the script to your 'schema_unmapped.yaml' to generate 'human_documentation.yaml'.
Successfully generated clean human blueprint at: human_documentation.yaml
